# NEO: NeRF It Once, Edit It Many Times for Continuous Object Manipulation

**ArXivist-generated reproduction notebook**
Paper: [arXiv:2607.24538](https://arxiv.org/abs/2607.24538)
Generated: 2026-07-28

This notebook walks through the key components of the implementation, runs a small-scale
training + editing loop on a **synthetic stand-in scene**, and verifies that the setup behaves
sensibly (losses decrease, shapes match) on a mini-dataset. See `README.md` for the full
Reproducibility Notes on what is faithfully implemented vs. substituted (real NEO-Dataset,
Stable Diffusion, CLIP/DINO, and AnyGrasp are all unreachable in this sandbox and use
documented, swappable stand-ins).

In [ ]:
# Cell 1 — Environment check
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU -- this notebook's demos use tiny synthetic data so this is fine, "
          "just slower than a GPU run of the full config.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Cell 2 — Install the project in editable mode (run once; safe to re-run)
import subprocess, sys, os
repo_root = os.path.dirname(os.getcwd())  # notebooks/ -> repo root
try:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-e", repo_root],
                            capture_output=True, text=True, timeout=300)
    print(result.stdout[-1000:] if result.returncode == 0 else result.stderr[-2000:])
except Exception as e:
    print(f"Install step failed non-fatally ({e}); if neo_nerf_editing is already importable "
          f"(e.g. you're running inside the repo's own venv), you can ignore this and continue.")

sys.path.insert(0, os.path.join(repo_root, "src"))
import neo_nerf_editing
print("neo_nerf_editing importable:", neo_nerf_editing.__name__)

## What this paper does

NEO edits a NeRF's *weights* directly so a robot can remove, inpaint, and relocate objects in a
scene — and predict what it will look like afterward — **without re-scanning**. Given a language
prompt and a planned 6-DoF motion, NEO:

1. **(Sec. II-A)** Removes the target object via *neural field resampling*: ray/box intersection
   segments are excluded from sampling entirely (rather than density-suppressed, like prior work
   DFF), avoiding the floating-point artifacts that under-sampling causes behind the object.
2. **(Sec. II-B)** Fills in newly-revealed, previously-unobserved surfaces via multiview-consistent
   progressive diffusion inpainting, training a mask-conditioned NeRF on the result.
3. **(Sec. II-C)** Distills the original and edited NeRFs into one persistent network via a
   region-wise teacher-student scheme, so edits compose across a sequence of manipulations.
4. **(Sec. II-D/E)** Updates the language field and repeats — the same (S',L') representation
   feeds back in as the input to the next edit.

This notebook demonstrates each numbered component against the repo's synthetic stand-in scene.

## Component 1 — The base joint scene+language NeRF ($F_\Theta$)

$$
(\mathcal{S}, \mathcal{L}) = F_{\Theta}(\mathcal{I}, \Omega) \qquad \text{(Eq. 1)}
$$

A positional-encoding MLP mapping 3D points + view directions to (density, RGB, language
feature). SIR note: the paper never states the backbone width/depth -- this repo uses
config-driven defaults (`configs/config.yaml`, marked `# ASSUMED`).

In [ ]:
from neo_nerf_editing.models.nerf_field import JointNeRFField
import torch

# Small config for demo purposes (full config.yaml uses hidden_dim=128, num_layers=6)
field = JointNeRFField(d_l=16, hidden_dim=32, num_layers=4, posenc_num_freqs=6,
                        posenc_include_input=True, skip_layers=[2])
print(field)

points = torch.randn(8, 5, 3)        # [rays=8, samples=5, xyz]
view_dirs = torch.randn(8, 1, 3)
out = field(points, view_dirs)
print("density:", out["density"].shape, " (expected [8, 5, 1])")
print("rgb:    ", out["rgb"].shape, "     (expected [8, 5, 3])")
print("lang:   ", out["lang_feat"].shape, "   (expected [8, 5, 16])")

## Component 2 — Neural field resampling for object removal (Sec. II-A, Fig. 4)

For each ray, the box $b$'s intersection segment is *excluded* from sampling, and the remaining
intervals are concatenated into one 1D domain before uniform + PDF-based hierarchical resampling
-- this is the paper's first core contribution, and it is implemented exactly as described.

In [ ]:
from neo_nerf_editing.models.object_removal import RayBoxExcluder, TwoStageResampler
from neo_nerf_editing.models.language_field import OrientedBox
import numpy as np

box = OrientedBox(center=np.array([0.0, 0.0, 0.0], dtype=np.float32), yaw=0.0,
                   extents=np.array([0.3, 0.3, 0.3], dtype=np.float32))
rays_o = torch.zeros(6, 3)
rays_d = torch.nn.functional.normalize(torch.tensor([[0,0,1.0]]).repeat(6,1) +
                                        0.05*torch.randn(6,3), dim=-1)

excluder, resampler = RayBoxExcluder(), TwoStageResampler()
hits = excluder.intersect(rays_o, rays_d, box, near=0.5, far=3.0)
print("ray/box hits (t_enter, t_exit), NaN = miss:\n", hits)

seg_near, seg_far = excluder.build_exclusion_intervals(0.5, 3.0, hits)
t_uniform = resampler.uniform_resample(seg_near, seg_far, n_uniform=16)
print("\nuniform-resampled t-values for ray 0 (should skip the box interval):\n", t_uniform[0])
inside = box.contains_torch(rays_o[:, None, :] + t_uniform[..., None] * rays_d[:, None, :])
print("\nany resampled point landed inside the excluded box?", bool(inside.any()))

## Component 3 — Language grounding (Sec. II, bounding-box retrieval)

$$
\phi(p) \in \mathbb{R}^{d_l}, \quad \mathcal{L}(\mathbf{x}) \in \mathbb{R}^{d_l}, \quad
\text{sim} = \cos(\phi(p), \mathcal{L}(\mathbf{x}))
$$

Cosine similarity over surface points, clustered, fit to an oriented box. **Note:** the real
CLIP-based embedder is unreachable here; `DeterministicHashEmbedder` is a semantically-inert
stand-in used only to exercise this code path (see `models/language_field.py` docstring and
`edit.py --localization` flag).

In [ ]:
from neo_nerf_editing.models.language_field import DeterministicHashEmbedder, LanguageGrounder

embedder = DeterministicHashEmbedder(d_l=16)
grounder = LanguageGrounder(embedder)

surface_points = torch.randn(200, 3) * 0.5
surface_lang_feats = torch.randn(200, 16)
prompt_embed = grounder.embed_prompt("move the soup can")

box_found = grounder.localize(surface_points, surface_lang_feats, prompt_embed, top_frac=0.1)
print("Localized (untrained, demo-only) box:", box_found)
print("\n(In edit.py, --localization oracle uses the synthetic scene's ground-truth object "
      "registry instead of this untrained language field, for a functionally meaningful demo.)")

## Component 4 — Losses (Eq. 3-6)

$$
\mathcal{L}_{rec} = \mathbb{E}_{(I,M,\Omega)\sim\mathcal{D}}\left[\ell(\mathcal{R}(\tilde{F}_\Theta,\Omega), I; M)\right]
\qquad
\mathcal{L}_{reg} = \frac{1}{n}\sum_i (\tilde{\sigma}_i - \tilde{\sigma}'_i)^2
$$

In [ ]:
from neo_nerf_editing.training.losses import MaskedTrainingObjective

objective = MaskedTrainingObjective()
rendered = torch.rand(32, 3)
target = torch.rand(32, 3)
mask = torch.rand(32) > 0.3
sigma = torch.rand(32, 5)
points = torch.randn(32, 5, 3) * 0.5

loss = objective(rendered, target, mask, sigma, points, box, lambda_rec=1.0, lambda_reg=1.0)
print("L_masked =", loss.item())

## Mini-training demo: base NeRF on the synthetic scene

Generates a tiny synthetic tabletop scene (no downloads), trains `JointNeRFField` for a
handful of steps, and confirms the reconstruction loss decreases.

In [ ]:
from neo_nerf_editing.data.synthetic_scene import ToySceneGenerator
from neo_nerf_editing.data.rays import fixed_scan_trajectory
from neo_nerf_editing.training.trainer import NeRFTrainer

scene_gen = ToySceneGenerator()
scene = scene_gen.generate_scene(seed=0, objects=["soup_can", "lego_brick"])
trajectory = fixed_scan_trajectory(n_views=4, radius=4.0, height=2.0)
dataset = scene_gen.render_views(scene, trajectory, H=24, W=24, near=2.0, far=8.0)
dataset_t = {"rays_o": dataset["rays_o"], "rays_d": dataset["rays_d"], "rgb": dataset["rgb_flat"]}
print("dataset rays:", dataset_t["rays_o"].shape[0])

mini_field = JointNeRFField(d_l=16, hidden_dim=32, num_layers=4, posenc_num_freqs=6,
                             posenc_include_input=True, skip_layers=[2])
trainer = NeRFTrainer(lr=5e-3, log_every=2)

def log_fn(step, loss):
    print(f"  step {step:3d} | loss {loss:.5f}")

history = trainer.fit(mini_field, dataset_t, n_steps=10, near=2.0, far=8.0,
                       n_samples=24, batch_size=128, device="cpu", log_fn=log_fn)
print(f"\nLoss went from {history['loss'][0]:.5f} to {history['loss'][-1]:.5f}")
assert history["loss"][-1] < history["loss"][0], "expected loss to decrease over 10 steps"
print("Sanity check passed: loss decreased.")

## Paper's reported results (Table I, object removal — 5 real NEO-Dataset scenes)

These are the numbers the paper itself reports; they are **not** reproduced here (the real
dataset/models are unavailable, see README). Use `evaluate.py` on this repo's synthetic scene for
this repo's own (much smaller-scale) measured numbers, and see `comparison/comparison_report.md`
for the honest side-by-side.

In [ ]:
paper_results = {
    "dataset": "NEO-Dataset (5 scenes, object removal)",
    "metric": "PSNR (Out region)",
    "reported_value": 25.43,
    "baseline": "DFF",
    "baseline_value": 21.64,
}
print("Paper's claimed results:")
for k, v in paper_results.items():
    print(f"  {k}: {v}")
print("\nTo reproduce (on this repo's synthetic stand-in scene), run:")
print("  python train.py --config configs/config.yaml")
print("  python edit.py --checkpoint runs/base_nerf/base_nerf.pt --prompt '...' --motion x,y,z,r,p,y")
print("  python evaluate.py --edited-checkpoint ... --edit-metadata ...")
print("Then see comparison/comparison_report.md for Stage 6's side-by-side analysis.")

## What to do next

1. **Full training**: `python train.py --config configs/config.yaml`
2. **Full edit + evaluation**: `python edit.py ...` then `python evaluate.py ...`
3. **Ablations (Table III)**: `python run_ablation.py --config configs/config.yaml`
4. **Compare results**: see `comparison/comparison_report.md` (Stage 6)

**Top implementation assumptions from the SIR** (see `sir-registry/arxiv_2607_24538/sir.json` for the full list):
- Optimizer/LR/batch size/step counts: **not stated in the paper** (confidence 0.45) — standard NeRF-literature defaults used
- Loss weights λ_rec, λ_reg: **not stated** (confidence 0.4) — assumed 1.0 each
- Base NeRF backbone width/depth: **not stated** (confidence ~0.5) — plain positional-encoding MLP, config-driven